<a href="https://colab.research.google.com/github/LINWOO0099/Categorical-Encoding/blob/api/main_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================
# TASK 1 — LOAD AND EXPLORE THE DATASET
# File: task1_load_explore.py
# ============================================

# Import pandas
import pandas as pd

# Load dataset
df = pd.read_csv("churnguard_data.csv")

# ============================================
# DATASET SHAPE
# ============================================

print("===== DATASET SHAPE =====")
print(df.shape)

# ============================================
# FIRST 5 ROWS
# ============================================

print("\n===== FIRST 5 ROWS =====")
print(df.head())

# ============================================
# COLUMN NAMES AND DATA TYPES
# ============================================

print("\n===== DATASET INFO =====")
print(df.info())

# ============================================
# MISSING VALUES
# ============================================

print("\n===== MISSING VALUES COUNT =====")
print(df.isnull().sum())

# ============================================
# DUPLICATE ROWS
# ============================================

print("\n===== DUPLICATE ROWS =====")
print(df.duplicated().sum())

# ============================================
# CHURN VALUE COUNTS
# ============================================

print("\n===== CHURN VALUE COUNTS =====")
print(df["Churn"].value_counts())

# ============================================
# UNIQUE VALUES IN CONTRACT COLUMN
# ============================================

print("\n===== UNIQUE CONTRACT VALUES =====")
print(df["Contract"].unique())

===== DATASET SHAPE =====
(1030, 12)

===== FIRST 5 ROWS =====
  customerID  gender  SeniorCitizen  tenure PhoneService InternetService  \
0  CUST-0032    Male              0    21.0          YES     Fiber optic   
1  CUST-0110    Male              0    55.0          YES     Fiber optic   
2  CUST-0137  Female              1    46.0          Yes     Fiber optic   
3  CUST-0089  Female              1    63.0          Yes     Fiber optic   
4  CUST-0919  Female              0     8.0          Yes             DSl   

         Contract PaperlessBilling     PaymentMethod  MonthlyCharges  \
0  Month-to-month               No     Credit card             29.73   
1        Two year              Yes     Bank transfer           46.32   
2  Month-to-month               No    Mailed check             87.06   
3  Month-to-month              YES      Mailed check           56.97   
4  month to month               No  Electronic check           39.69   

  TotalCharges Churn  
0       600.01   Yes  
1

In [3]:
# ============================================
# TASK 2 — CLEAN THE DATASET
# File: task2_clean_data.py
# ============================================

# Import pandas
import pandas as pd
import numpy as np

# ============================================
# LOAD DATASET
# ============================================

df = pd.read_csv("churnguard_data.csv")

# ============================================
# DROP customerID COLUMN
# ============================================

df = df.drop("customerID", axis=1)

# ============================================
# REMOVE DUPLICATE ROWS
# ============================================

df = df.drop_duplicates()

# ============================================
# STRIP WHITESPACE
# ============================================

df["gender"] = df["gender"].str.strip()
df["PaymentMethod"] = df["PaymentMethod"].str.strip()

# ============================================
# STANDARDISE CASING
# ============================================

df["Churn"] = df["Churn"].str.strip().str.title()
df["PhoneService"] = df["PhoneService"].str.strip().str.title()
df["PaperlessBilling"] = df["PaperlessBilling"].str.strip().str.title()

# ============================================
# FIX CONTRACT COLUMN
# ============================================

contract_mapping = {
    "monthly": "Month-to-month",
    "month to month": "Month-to-month",
    "month-to-month": "Month-to-month",

    "1 year": "One year",
    "one year": "One year",

    "2 year": "Two year",
    "two year": "Two year"
}

df["Contract"] = (
    df["Contract"]
    .str.strip()
    .str.lower()
    .replace(contract_mapping)
)

# ============================================
# FIX INTERNETSERVICE COLUMN
# ============================================

internet_mapping = {
    "dsl": "DSL",

    "fibre optic": "Fiber optic",
    "fiberoptic": "Fiber optic",
    "fiber optic": "Fiber optic",

    "none": "No",
    "no": "No"
}

df["InternetService"] = (
    df["InternetService"]
    .str.strip()
    .str.lower()
    .replace(internet_mapping)
)

# ============================================
# FIX TotalCharges COLUMN
# ============================================

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

# ============================================
# REMOVE INVALID tenure VALUES
# ============================================

df = df[df["tenure"] > 0]

# ============================================
# REMOVE OUTLIERS IN MonthlyCharges
# ============================================

df = df[
    (df["MonthlyCharges"] >= 10) &
    (df["MonthlyCharges"] <= 200)
]

# ============================================
# FILL MISSING VALUES
# ============================================

# MonthlyCharges → mean
monthly_mean = df["MonthlyCharges"].mean()
df["MonthlyCharges"] = df["MonthlyCharges"].fillna(monthly_mean)

# TotalCharges → mean
total_mean = df["TotalCharges"].mean()
df["TotalCharges"] = df["TotalCharges"].fillna(total_mean)

# tenure → median with integer rounding
tenure_median = round(df["tenure"].median())
df["tenure"] = df["tenure"].fillna(tenure_median)

# ============================================
# PRINT CLEANED DATAFRAME SHAPE
# ============================================

print("===== CLEANED DATAFRAME SHAPE =====")
print(df.shape)

# ============================================
# PRINT MISSING VALUE COUNTS
# ============================================

print("\n===== MISSING VALUES COUNT =====")
print(df.isnull().sum())

===== CLEANED DATAFRAME SHAPE =====
(867, 11)

===== MISSING VALUES COUNT =====
gender               0
SeniorCitizen        0
tenure               0
PhoneService         0
InternetService     14
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges         0
Churn                0
dtype: int64


In [4]:
# ============================================
# TASK 3 — TRAIN A CLASSIFICATION MODEL
# File: task3_train_model.py
# ============================================

# Import libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# ============================================
# LOAD DATASET
# ============================================

df = pd.read_csv("churnguard_data.csv")

# ============================================
# TASK 2 CLEANING STEPS
# ============================================

# Drop customerID
df = df.drop("customerID", axis=1)

# Remove duplicates
df = df.drop_duplicates()

# Strip whitespace
df["gender"] = df["gender"].str.strip()
df["PaymentMethod"] = df["PaymentMethod"].str.strip()

# Standardize casing
df["Churn"] = df["Churn"].str.strip().str.title()
df["PhoneService"] = df["PhoneService"].str.strip().str.title()
df["PaperlessBilling"] = df["PaperlessBilling"].str.strip().str.title()

# ============================================
# FIX CONTRACT COLUMN
# ============================================

contract_mapping = {
    "monthly": "Month-to-month",
    "month to month": "Month-to-month",
    "month-to-month": "Month-to-month",

    "1 year": "One year",
    "one year": "One year",

    "2 year": "Two year",
    "two year": "Two year"
}

df["Contract"] = (
    df["Contract"]
    .str.strip()
    .str.lower()
    .replace(contract_mapping)
)

# ============================================
# FIX INTERNETSERVICE COLUMN
# ============================================

internet_mapping = {
    "dsl": "DSL",

    "fibre optic": "Fiber optic",
    "fiberoptic": "Fiber optic",
    "fiber optic": "Fiber optic",

    "none": "No",
    "no": "No"
}

df["InternetService"] = (
    df["InternetService"]
    .str.strip()
    .str.lower()
    .replace(internet_mapping)
)

# ============================================
# FIX TotalCharges COLUMN
# ============================================

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

# ============================================
# REMOVE INVALID ROWS
# ============================================

# Remove invalid tenure
df = df[df["tenure"] > 0]

# Remove MonthlyCharges outliers
df = df[
    (df["MonthlyCharges"] >= 10) &
    (df["MonthlyCharges"] <= 200)
]

# ============================================
# FILL MISSING VALUES
# ============================================

# MonthlyCharges → mean
monthly_mean = df["MonthlyCharges"].mean()
df["MonthlyCharges"] = df["MonthlyCharges"].fillna(monthly_mean)

# TotalCharges → mean
total_mean = df["TotalCharges"].mean()
df["TotalCharges"] = df["TotalCharges"].fillna(total_mean)

# tenure → median
tenure_median = round(df["tenure"].median())
df["tenure"] = df["tenure"].fillna(tenure_median)

# ============================================
# ENCODE TARGET COLUMN
# ============================================

df["Churn"] = df["Churn"].map({
    "Yes": 1,
    "No": 0
})

# ============================================
# ONE-HOT ENCODING
# ============================================

categorical_columns = [
    "gender",
    "PhoneService",
    "InternetService",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod"
]

df = pd.get_dummies(
    df,
    columns=categorical_columns,
    drop_first=True
)

# ============================================
# SPLIT FEATURES AND TARGET
# ============================================

X = df.drop("Churn", axis=1)
y = df["Churn"]

# ============================================
# TRAIN-TEST SPLIT
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ============================================
# TRAIN LOGISTIC REGRESSION MODEL
# ============================================

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

# ============================================
# PREDICTIONS
# ============================================

y_pred = model.predict(X_test)

# ============================================
# EVALUATION
# ============================================

accuracy = accuracy_score(y_test, y_pred)

print("===== ACCURACY SCORE =====")
print(accuracy)

print("\n===== CLASSIFICATION REPORT =====")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Stay", "Churn"]
    )
)

===== ACCURACY SCORE =====
0.6954022988505747

===== CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

        Stay       0.75      0.85      0.80       122
       Churn       0.49      0.33      0.39        52

    accuracy                           0.70       174
   macro avg       0.62      0.59      0.59       174
weighted avg       0.67      0.70      0.68       174



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [5]:
# ============================================
# TASK 4 — BUILD A PREDICTION SCRIPT
# File: task4_predict.py
# ============================================

# Import libraries
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression

# ============================================
# LOAD DATASET
# ============================================

df = pd.read_csv("churnguard_data.csv")

# ============================================
# TASK 2 CLEANING STEPS
# ============================================

# Drop customerID
df = df.drop("customerID", axis=1)

# Remove duplicates
df = df.drop_duplicates()

# Strip whitespace
df["gender"] = df["gender"].str.strip()
df["PaymentMethod"] = df["PaymentMethod"].str.strip()

# Standardize casing
df["Churn"] = df["Churn"].str.strip().str.title()
df["PhoneService"] = df["PhoneService"].str.strip().str.title()
df["PaperlessBilling"] = df["PaperlessBilling"].str.strip().str.title()

# ============================================
# FIX CONTRACT COLUMN
# ============================================

contract_mapping = {
    "monthly": "Month-to-month",
    "month to month": "Month-to-month",
    "month-to-month": "Month-to-month",

    "1 year": "One year",
    "one year": "One year",

    "2 year": "Two year",
    "two year": "Two year"
}

df["Contract"] = (
    df["Contract"]
    .str.strip()
    .str.lower()
    .replace(contract_mapping)
)

# ============================================
# FIX INTERNETSERVICE COLUMN
# ============================================

internet_mapping = {
    "dsl": "DSL",

    "fibre optic": "Fiber optic",
    "fiberoptic": "Fiber optic",
    "fiber optic": "Fiber optic",

    "none": "No",
    "no": "No"
}

df["InternetService"] = (
    df["InternetService"]
    .str.strip()
    .str.lower()
    .replace(internet_mapping)
)

# ============================================
# FIX TotalCharges COLUMN
# ============================================

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

# ============================================
# REMOVE INVALID ROWS
# ============================================

# Remove invalid tenure
df = df[df["tenure"] > 0]

# Remove MonthlyCharges outliers
df = df[
    (df["MonthlyCharges"] >= 10) &
    (df["MonthlyCharges"] <= 200)
]

# ============================================
# FILL MISSING VALUES
# ============================================

# MonthlyCharges → mean
monthly_mean = df["MonthlyCharges"].mean()
df["MonthlyCharges"] = df["MonthlyCharges"].fillna(monthly_mean)

# TotalCharges → mean
total_mean = df["TotalCharges"].mean()
df["TotalCharges"] = df["TotalCharges"].fillna(total_mean)

# tenure → median
tenure_median = round(df["tenure"].median())
df["tenure"] = df["tenure"].fillna(tenure_median)

# ============================================
# ENCODE CHURN
# ============================================

df["Churn"] = df["Churn"].map({
    "Yes": 1,
    "No": 0
})

# ============================================
# ENCODE CONTRACT COLUMN
# ============================================

contract_encoding = {
    "Month-to-month": 0,
    "One year": 1,
    "Two year": 2
}

df["Contract"] = df["Contract"].map(contract_encoding)

# ============================================
# SELECT FEATURES
# ============================================

X = df[
    [
        "tenure",
        "MonthlyCharges",
        "TotalCharges",
        "SeniorCitizen",
        "Contract"
    ]
]

y = df["Churn"]

# ============================================
# TRAIN MODEL ON FULL DATASET
# ============================================

model = LogisticRegression(max_iter=1000)

model.fit(X, y)

# ============================================
# USER INPUT
# ============================================

tenure = int(input("Enter tenure (months): "))

monthly_charges = float(
    input("Enter Monthly Charges: ")
)

total_charges = float(
    input("Enter Total Charges: ")
)

senior_citizen = int(
    input("Senior Citizen? (1 = Yes, 0 = No): ")
)

contract = int(
    input(
        "Contract type (0 = Month-to-month, 1 = One year, 2 = Two year): "
    )
)

# ============================================
# CREATE INPUT DATAFRAME
# ============================================

new_customer = pd.DataFrame(
    [[
        tenure,
        monthly_charges,
        total_charges,
        senior_citizen,
        contract
    ]],
    columns=[
        "tenure",
        "MonthlyCharges",
        "TotalCharges",
        "SeniorCitizen",
        "Contract"
    ]
)

# ============================================
# MAKE PREDICTION
# ============================================

prediction = model.predict(new_customer)[0]

# ============================================
# PRINT RESULT
# ============================================

if prediction == 1:
    print(
        "\nPrediction: This customer is likely to CHURN."
    )

else:
    print(
        "\nPrediction: This customer is likely to STAY."
    )

Enter tenure (months): 12
Enter Monthly Charges: 45
Enter Total Charges: 324
Senior Citizen? (1 = Yes, 0 = No): 1
Contract type (0 = Month-to-month, 1 = One year, 2 = Two year): 9

Prediction: This customer is likely to STAY.


## Training a Simple ML Model Example

This section demonstrates how to train a basic Logistic Regression model on the processed dataset.

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Load dataset
df_model_example = pd.read_csv("churnguard_data.csv")

# --- Data Cleaning and Preprocessing (Reusing steps from Task 2 and 3) ---

# Drop customerID
df_model_example = df_model_example.drop("customerID", axis=1)

# Remove duplicates
df_model_example = df_model_example.drop_duplicates()

# Strip whitespace
df_model_example["gender"] = df_model_example["gender"].str.strip()
df_model_example["PaymentMethod"] = df_model_example["PaymentMethod"].str.strip()

# Standardize casing
df_model_example["Churn"] = df_model_example["Churn"].str.strip().str.title()
df_model_example["PhoneService"] = df_model_example["PhoneService"].str.strip().str.title()
df_model_example["PaperlessBilling"] = df_model_example["PaperlessBilling"].str.strip().str.title()

# Fix Contract column
contract_mapping = {
    "monthly": "Month-to-month",
    "month to month": "Month-to-month",
    "month-to-month": "Month-to-month",
    "1 year": "One year",
    "one year": "One year",
    "2 year": "Two year",
    "two year": "Two year"
}
df_model_example["Contract"] = (
    df_model_example["Contract"]
    .str.strip()
    .str.lower()
    .replace(contract_mapping)
)

# Fix InternetService column
internet_mapping = {
    "dsl": "DSL",
    "fibre optic": "Fiber optic",
    "fiberoptic": "Fiber optic",
    "fiber optic": "Fiber optic",
    "none": "No",
    "no": "No"
}
df_model_example["InternetService"] = (
    df_model_example["InternetService"]
    .str.strip()
    .str.lower()
    .replace(internet_mapping)
)

# Fix TotalCharges column
df_model_example["TotalCharges"] = pd.to_numeric(
    df_model_example["TotalCharges"],
    errors="coerce"
)

# Remove invalid tenure
df_model_example = df_model_example[df_model_example["tenure"] > 0]

# Remove MonthlyCharges outliers
df_model_example = df_model_example[
    (df_model_example["MonthlyCharges"] >= 10) &
    (df_model_example["MonthlyCharges"] <= 200)
]

# Fill missing values
monthly_mean = df_model_example["MonthlyCharges"].mean()
df_model_example["MonthlyCharges"] = df_model_example["MonthlyCharges"].fillna(monthly_mean)

total_mean = df_model_example["TotalCharges"].mean()
df_model_example["TotalCharges"] = df_model_example["TotalCharges"].fillna(total_mean)

tenure_median = round(df_model_example["tenure"].median())
df_model_example["tenure"] = df_model_example["tenure"].fillna(tenure_median)

# Encode target column
df_model_example["Churn"] = df_model_example["Churn"].map({
    "Yes": 1,
    "No": 0
})

# --- Feature Engineering: One-Hot Encoding ---
categorical_cols_for_ohe = [
    "gender",
    "PhoneService",
    "InternetService",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod"
]

df_model_example = pd.get_dummies(
    df_model_example,
    columns=categorical_cols_for_ohe,
    drop_first=True # Avoid multicollinearity
)

# --- Model Training ---

# Split features (X) and target (y)
X_example = df_model_example.drop("Churn", axis=1)
y_example = df_model_example["Churn"]

# Train-test split
X_train_example, X_test_example, y_train_example, y_test_example = train_test_split(
    X_example,
    y_example,
    test_size=0.2,
    random_state=42 # for reproducibility
)

# Initialize and train a Logistic Regression model
model_example = LogisticRegression(max_iter=1000) # Increased max_iter to help with convergence
model_example.fit(X_train_example, y_train_example)

# --- Prediction and Evaluation ---

y_pred_example = model_example.predict(X_test_example)

print("\n===== MODEL ACCURACY ====")
print(f"Accuracy: {accuracy_score(y_test_example, y_pred_example):.4f}")

print("\n===== CLASSIFICATION REPORT ====")
print(classification_report(y_test_example, y_pred_example, target_names=["Stay", "Churn"]))


===== MODEL ACCURACY ====
Accuracy: 0.6954

===== CLASSIFICATION REPORT ====
              precision    recall  f1-score   support

        Stay       0.75      0.85      0.80       122
       Churn       0.49      0.33      0.39        52

    accuracy                           0.70       174
   macro avg       0.62      0.59      0.59       174
weighted avg       0.67      0.70      0.68       174



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
